# 00 · Análisis Exploratorio de Datos (EDA)
**EduPredict** · Samsung Innovation Campus 2025 · Reto 4 · Universidad del Rosario

> **Responsable:** Angela Yineth Quiñones Martinez  
> **Objetivo:** Comprender en profundidad el dataset de evaluaciones docentes antes de modelar.

---

In [ ]:
# ── Importaciones ──────────────────────────────────────────────────
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
from collections import Counter

# Paleta Universidad del Rosario (Manual de Marca 2020)
UR_RED    = "#DA0921"   # Rojo Mutisia Clematis - color institucional
UR_NAVY   = "#242839"   # Azul muy oscuro
UR_BLUE   = "#3100A0"   # Azul complementario
UR_TECH   = "#0E6A8C"   # Azul tecnología
UR_GREEN  = "#1A6E3A"   # Verde positivo
UR_GRAY   = "#6B6B6B"
UR_LIGHT  = "#F7F4F4"

COLORS_CLASE = {"Mejora": UR_GREEN, "Estable": UR_TECH, "En riesgo": UR_RED}
ORDER = ["Mejora", "Estable", "En riesgo"]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#F8F8F8",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "font.size": 11,
})

print("✅ Importaciones correctas")

## 1 · Carga y validación del dataset

In [ ]:
from src.preprocessing import load_and_validate, engineer_features

df_raw = load_and_validate("../data/evaluaciones_docentes.csv")
df = engineer_features(df_raw)

print(f"Shape: {df.shape}")
print(f"\nPrimeras filas:")
df.head(3)

In [ ]:
# Resumen estadístico completo
print("=== ESTADÍSTICAS DESCRIPTIVAS ===")
df[["puntaje_claridad","puntaje_metodologia","puntaje_evaluacion",
    "numero_estudiantes","puntaje_promedio"]].describe().round(3)

In [ ]:
# Calidad de datos
print("Valores nulos:", df.isnull().sum().sum())
print("Duplicados:   ", df.duplicated().sum())
print("\nDocentes únicos:", df["id_docente"].nunique())
print("Asignaturas:    ", df["asignatura"].nunique())
print("Semestres:      ", sorted(df["semestre"].unique()))

## 2 · Distribución de la variable objetivo

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Donut chart
dist = df["tendencia_desempeno"].value_counts()
sizes = [dist.get(o, 0) for o in ORDER]
cols  = [COLORS_CLASE[o] for o in ORDER]
wedges, _, autotexts = axes[0].pie(
    sizes, autopct="%1.1f%%", colors=cols, startangle=90,
    wedgeprops=dict(width=0.55, edgecolor="white", linewidth=2),
    pctdistance=0.75,
)
for at in autotexts:
    at.set_fontsize(12); at.set_color("white"); at.set_fontweight("bold")
axes[0].set_title("Distribución de clases", fontsize=13, fontweight="bold", color=UR_NAVY)
patches = [mpatches.Patch(color=COLORS_CLASE[o], label=f"{o}  ({dist.get(o,0)})") for o in ORDER]
axes[0].legend(handles=patches, loc="lower center", bbox_to_anchor=(0.5, -0.18), frameon=False)

# Barras por semestre apiladas
pivot = df.groupby(["semestre","tendencia_desempeno"]).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=ORDER, fill_value=0)
bottom = np.zeros(len(pivot))
for col in ORDER:
    axes[1].bar(pivot.index, pivot[col], bottom=bottom,
                color=COLORS_CLASE[col], label=col, edgecolor="white", linewidth=0.5)
    bottom += pivot[col].values
axes[1].set_title("Evolución por semestre", fontsize=13, fontweight="bold", color=UR_NAVY)
axes[1].tick_params(axis="x", rotation=35)
axes[1].legend(frameon=False)

plt.suptitle("EduPredict - Variable objetivo", fontsize=14, fontweight="bold",
             color=UR_RED, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/eda_01_target.png", dpi=150, bbox_inches="tight")
plt.show()
print("\n📌 Desbalance leve: 50/30/20 -> se usará class_weight='balanced' en RF y F1-macro como métrica principal")

## 3 · Análisis de puntajes por clase

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
score_cols = ["puntaje_claridad", "puntaje_metodologia", "puntaje_evaluacion"]
labels_short = ["Claridad", "Metodología", "Evaluación"]

for i, (col, lab) in enumerate(zip(score_cols, labels_short)):
    data_box = [df[df["tendencia_desempeno"] == c][col].values for c in ORDER]
    bp = axes[i].boxplot(data_box, patch_artist=True, notch=False,
                         medianprops=dict(color="white", linewidth=2.5))
    for patch, c in zip(bp["boxes"], [COLORS_CLASE[o] for o in ORDER]):
        patch.set_facecolor(c); patch.set_alpha(0.85)
    for el in ["whiskers", "caps", "fliers"]:
        for item in bp[el]: item.set_color(UR_GRAY)
    axes[i].set_xticks([1, 2, 3])
    axes[i].set_xticklabels(ORDER, fontsize=9)
    axes[i].set_title(f"Puntaje {lab}", fontsize=12, fontweight="bold", color=UR_NAVY)
    axes[i].set_ylabel("Puntaje (1-5)", color=UR_GRAY)

plt.suptitle("Distribución de puntajes por clase", fontsize=14, fontweight="bold",
             color=UR_RED, y=1.01)
plt.tight_layout()
plt.savefig("../outputs/eda_02_scores.png", dpi=150, bbox_inches="tight")
plt.show()

# Tabla de medias
print("Medias por clase:")
df.groupby("tendencia_desempeno")[score_cols + ["puntaje_promedio"]].mean().round(3)

## 4 · Análisis de correlaciones

In [ ]:
corr_cols = ["puntaje_claridad","puntaje_metodologia","puntaje_evaluacion",
             "numero_estudiantes","puntaje_promedio"]
corr = df[corr_cols].corr().round(3)

fig, ax = plt.subplots(figsize=(7, 5))
cmap = LinearSegmentedColormap.from_list("ur", [UR_RED, "white", UR_GREEN])
im = ax.imshow(corr.values, cmap=cmap, vmin=-1, vmax=1)
labels_corr = ["Claridad","Metodología","Evaluación","N°Estud.","Promedio"]
ax.set_xticks(range(5)); ax.set_xticklabels(labels_corr, rotation=30, ha="right")
ax.set_yticks(range(5)); ax.set_yticklabels(labels_corr)
for i in range(5):
    for j in range(5):
        v = corr.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=9,
                color="white" if abs(v) > 0.5 else UR_NAVY, fontweight="bold")
plt.colorbar(im, ax=ax, shrink=0.85)
ax.set_title("Matriz de correlación", fontsize=13, fontweight="bold", color=UR_NAVY)
plt.tight_layout()
plt.savefig("../outputs/eda_03_corr.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📌 numero_estudiantes tiene r=-0.03 con puntaje_promedio -> feature débil pero lo incluimos")

## 5 · Análisis de comentarios (entrada CNN 1D)

In [ ]:
df["len_comentario"] = df["comentario"].str.len()
df["num_palabras"]   = df["comentario"].str.split().str.len()

print("Longitud de comentarios (caracteres):")
print(df["len_comentario"].describe().round(1))

print("\nTop 10 comentarios:")
for i, (com, n) in enumerate(Counter(df["comentario"]).most_common(10), 1):
    print(f"  {i:2}. [{n:3}x · {n/len(df)*100:.1f}%] {com}")

In [ ]:
# Señales léxicas por clase
positive_kw = ["excelente","muy claro","dinámica","entretenida","paciencia","útiles","inspiró","sabe mucho"]
negative_kw  = ["aburrida","monótona","no explica","perdido","no resuelve","falta dinamismo","parciales no"]

fig, ax = plt.subplots(figsize=(10, 5))
pos_counts, neg_counts = [], []
for c in ORDER:
    sub = df[df["tendencia_desempeno"] == c]["comentario"].str.lower()
    pos_counts.append(sum(sub.str.contains("|".join(positive_kw))))
    neg_counts.append(sum(sub.str.contains("|".join(negative_kw))))

x = np.arange(len(ORDER)); w = 0.35
ax.bar(x - w/2, pos_counts, w, label="Positivos", color=UR_GREEN, alpha=0.9, edgecolor="white")
ax.bar(x + w/2, neg_counts, w, label="Negativos", color=UR_RED, alpha=0.9, edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(ORDER)
ax.set_title("Señales léxicas por clase", fontsize=13, fontweight="bold", color=UR_NAVY)
ax.legend(frameon=False)
for i, (p, n) in enumerate(zip(pos_counts, neg_counts)):
    ax.text(i - w/2, p + 5, str(p), ha="center", fontsize=10, color=UR_GREEN, fontweight="bold")
    ax.text(i + w/2, n + 5, str(n), ha="center", fontsize=10, color=UR_RED, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/eda_04_text.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n📌 'En riesgo' concentra el 36% de señales negativas vs 8% en 'Mejora'")
print("📌 El vocabulario limitado (~20 frases) es ideal para embedding entrenable - no se necesita FastText")

## 6 · Análisis por docente

In [ ]:
doc_stats = df.groupby("id_docente").agg(
    evaluaciones=("id_docente","count"),
    prom_promedio=("puntaje_promedio","mean"),
    pct_riesgo=("tendencia_desempeno", lambda x: (x == "En riesgo").mean() * 100)
).round(2)

print("Top 5 docentes con mayor % En riesgo:")
print(doc_stats.sort_values("pct_riesgo", ascending=False).head(5))
print("\nTop 5 docentes con mejor desempeño:")
print(doc_stats.sort_values("prom_promedio", ascending=False).head(5))

## 7 · Conclusiones del EDA

| Hallazgo | Impacto en el modelo |
|---|---|
| 0 nulos, 0 duplicados | Dataset listo para entrenamiento directo |
| Desbalance leve 50/30/20 | Usar `class_weight='balanced'` y F1-macro |
| Brecha de puntajes: En riesgo=2.49 vs Mejora=4.46 | RF tendrá alta separabilidad numérica |
| `numero_estudiantes` r=-0.03 | Feature débil pero lo incluimos |
| ~20 frases únicas en comentarios | Embedding entrenable (no FastText) |
| 'En riesgo' 36% señales negativas | CNN 1D diferenciará bien las clases extremas |

> ✅ **EDA completado** - pasar a `01_preprocessing.ipynb`
